# 04 - Text Cleaning

Companion to [`../../data_cleaning/text_cleaning.md`](../../data_cleaning/text_cleaning.md).

---

## Why text cleaning matters

Raw text data is messy. Before analysis or modeling, you need to normalize it. Common issues:

- **Encoding artifacts** - accented characters, smart quotes, non-ASCII symbols
- **HTML/XML markup** - scraped text often contains tags
- **URLs and emails** - unique per record, add noise
- **Inconsistent casing** - "Great" vs "great" vs "GREAT"
- **Duplicate content** - same review posted multiple times
- **Special characters** - emojis, whitespace artifacts, control characters

In [1]:
import re, unicodedata
import pandas as pd

raw = pd.Series([
    "  Loved THE product!! Highly recommend :)  ",
    "I dont like it... too expensive",
    "AMAZING!!! Worth every penny.",
    "Visit https://example.com for more, email me at user@example.com",
    "cafe was great",
    "<p>HTML <b>markup</b> in the text</p>",
    "Loved THE product!! Highly recommend :)",
])
raw

0            Loved THE product!! Highly recommend :)  
1                      I dont like it... too expensive
2                        AMAZING!!! Worth every penny.
3    Visit https://example.com for more, email me a...
4                                       cafe was great
5                <p>HTML <b>markup</b> in the text</p>
6              Loved THE product!! Highly recommend :)
dtype: str

## Step-by-step cleaning pipeline

### 1. Decode and normalize unicode

NFC normalization converts composed characters (e-acute as one codepoint) and decomposed characters (e + combining accent as two codepoints) to a consistent form. This prevents "duplicate" strings that look identical but have different byte representations.

In [2]:
def nfc(s): return unicodedata.normalize('NFC', s)
result = raw.map(nfc)
for i, (r, c) in enumerate(zip(raw, result)):
    print(f"{i:2d}: {r!r:55s} -> {c!r}")


 0: '  Loved THE product!! Highly recommend :)  '           -> '  Loved THE product!! Highly recommend :)  '
 1: 'I dont like it... too expensive'                       -> 'I dont like it... too expensive'
 2: 'AMAZING!!! Worth every penny.'                         -> 'AMAZING!!! Worth every penny.'
 3: 'Visit https://example.com for more, email me at user@example.com' -> 'Visit https://example.com for more, email me at user@example.com'
 4: 'cafe was great'                                        -> 'cafe was great'
 5: '<p>HTML <b>markup</b> in the text</p>'                 -> '<p>HTML <b>markup</b> in the text</p>'
 6: 'Loved THE product!! Highly recommend :)'               -> 'Loved THE product!! Highly recommend :)'


### 2. Strip HTML tags

HTML tags are artifacts of web scraping. They add noise to text analysis.

In [3]:
def strip_html(s): return re.sub(r'<[^>]+>', ' ', s)
result = raw.map(strip_html)
for i, (r, c) in enumerate(zip(raw, result)):
    print(f"{i:2d}: {r!r:55s} -> {c!r}")


 0: '  Loved THE product!! Highly recommend :)  '           -> '  Loved THE product!! Highly recommend :)  '
 1: 'I dont like it... too expensive'                       -> 'I dont like it... too expensive'
 2: 'AMAZING!!! Worth every penny.'                         -> 'AMAZING!!! Worth every penny.'
 3: 'Visit https://example.com for more, email me at user@example.com' -> 'Visit https://example.com for more, email me at user@example.com'
 4: 'cafe was great'                                        -> 'cafe was great'
 5: '<p>HTML <b>markup</b> in the text</p>'                 -> ' HTML  markup  in the text '
 6: 'Loved THE product!! Highly recommend :)'               -> 'Loved THE product!! Highly recommend :)'


### 3. Replace URLs and emails with placeholders

URLs and emails are almost always unique per record. Replacing them with `<URL>` and `<EMAIL>` prevents them from dominating word counts.

In [4]:
def mask(s):
    s = re.sub(r'https?://\S+', '<URL>', s)
    s = re.sub(r'\S+@\S+', '<EMAIL>', s)
    return s
result = raw.map(mask)
for i, (r, c) in enumerate(zip(raw, result)):
    print(f"{i:2d}: {r!r:55s} -> {c!r}")


 0: '  Loved THE product!! Highly recommend :)  '           -> '  Loved THE product!! Highly recommend :)  '
 1: 'I dont like it... too expensive'                       -> 'I dont like it... too expensive'
 2: 'AMAZING!!! Worth every penny.'                         -> 'AMAZING!!! Worth every penny.'
 3: 'Visit https://example.com for more, email me at user@example.com' -> 'Visit <URL> for more, email me at <EMAIL>'
 4: 'cafe was great'                                        -> 'cafe was great'
 5: '<p>HTML <b>markup</b> in the text</p>'                 -> '<p>HTML <b>markup</b> in the text</p>'
 6: 'Loved THE product!! Highly recommend :)'               -> 'Loved THE product!! Highly recommend :)'


### 4. Whitespace and case normalization

Strip leading/trailing whitespace, collapse multiple spaces into one, and optionally lowercase.

In [5]:
def squash_ws(s): return re.sub(r'\s+', ' ', s).strip()
clean = raw.map(nfc).map(strip_html).map(mask).map(squash_ws)
clean

0      Loved THE product!! Highly recommend :)
1              I dont like it... too expensive
2                AMAZING!!! Worth every penny.
3    Visit <URL> for more, email me at <EMAIL>
4                               cafe was great
5                      HTML markup in the text
6      Loved THE product!! Highly recommend :)
dtype: str

### 5. Remove punctuation (optional)

Removing punctuation reduces vocabulary size. But be careful - it hurts sentiment models that rely on "!!!" or "..." as signals.

In [6]:
def remove_punct(s): return re.sub(r'[^\w\s]', '', s)
with_punct = clean.copy()
without_punct = clean.map(remove_punct)
for i in range(len(clean)):
    print(f"Original:  {with_punct.iloc[i]!r}")
    print(f"No punct:  {without_punct.iloc[i]!r}\n")

Original:  'Loved THE product!! Highly recommend :)'
No punct:  'Loved THE product Highly recommend '

Original:  'I dont like it... too expensive'
No punct:  'I dont like it too expensive'

Original:  'AMAZING!!! Worth every penny.'
No punct:  'AMAZING Worth every penny'

Original:  'Visit <URL> for more, email me at <EMAIL>'
No punct:  'Visit URL for more email me at EMAIL'

Original:  'cafe was great'
No punct:  'cafe was great'

Original:  'HTML markup in the text'
No punct:  'HTML markup in the text'

Original:  'Loved THE product!! Highly recommend :)'
No punct:  'Loved THE product Highly recommend '



### 6. Deduplicate (case-insensitive)

Near-duplicates waste compute and bias your model toward overrepresented content.

In [7]:
dedup_key = clean.str.lower()
duplicates = dedup_key[dedup_key.duplicated()]
print(f"Duplicates found: {len(duplicates)}")
deduped = clean[~dedup_key.duplicated()].reset_index(drop=True)
deduped

Duplicates found: 1


0      Loved THE product!! Highly recommend :)
1              I dont like it... too expensive
2                AMAZING!!! Worth every penny.
3    Visit <URL> for more, email me at <EMAIL>
4                               cafe was great
5                      HTML markup in the text
dtype: str

## Advanced: Emoji handling

Emojis carry sentiment but break traditional tokenizers. Options: keep them (modern models handle them), map to text, or drop them.

In [8]:
emoji_text = pd.Series([
    "Love this product! \U0001f60d\U0001f389",
    "Terrible experience \U0001f621\U0001f44e",
    "Meh, it's okay \U0001f610",
    "No emojis here, just text.",
])

# Detect emojis using Unicode ranges
def has_emoji(s):
    return bool(re.search(r'[\U0001F600-\U0001F9FF\U00002702-\U000027B0\U0001F300-\U0001F5FF]', s))

emoji_text.map(has_emoji)

0     True
1     True
2     True
3    False
dtype: bool

### Emoji to text mapping

For models that don't handle emojis well, map them to descriptive text.

In [9]:
emoji_map = {
    '😍': '[heart_eyes]',
    '🎉': '[party]',
    '😡': '[angry]',
    '👎': '[thumbs_down]',
    '😐': '[neutral]',
    '😀': '[smile]',
    '❤️': '[heart]',
}

def map_emojis(s):
    for emoji, text in emoji_map.items():
        s = s.replace(emoji, text)
    return s

print("Before:")
for t in emoji_text:
    print(f"  {t!r}")
print("\nAfter:")
for t in emoji_text.map(map_emojis):
    print(f"  {t!r}")

Before:
  'Love this product! 😍🎉'
  'Terrible experience 😡👎'
  "Meh, it's okay 😐"
  'No emojis here, just text.'

After:
  'Love this product! [heart_eyes][party]'
  'Terrible experience [angry][thumbs_down]'
  "Meh, it's okay [neutral]"
  'No emojis here, just text.'


## Advanced: Number normalization

Numbers in text (prices, dates, phone numbers) are often formatted inconsistently.

In [10]:
number_text = pd.Series([
    "It costs $1,234.56!",
    "Price: 1.234,56 EUR",
    "Call 555-123-4567",
    "Ordered on 01/15/2024",
])

def normalize_numbers(s):
    s = re.sub(r'\$[\d,]+\.?\d*', '<CURRENCY>', s)
    s = re.sub(r'[\d]+\.[\d]{3},[\d]+', '<CURRENCY>', s)
    s = re.sub(r'\b\d{3}-\d{3}-\d{4}\b', '<PHONE>', s)
    s = re.sub(r'\b\d{1,2}/\d{1,2}/\d{2,4}\b', '<DATE>', s)
    return s

for i in range(len(number_text)):
    print(f"{number_text.iloc[i]!r:35s} -> {normalize_numbers(number_text.iloc[i])!r}")

'It costs $1,234.56!'               -> 'It costs <CURRENCY>!'
'Price: 1.234,56 EUR'               -> 'Price: <CURRENCY> EUR'
'Call 555-123-4567'                 -> 'Call <PHONE>'
'Ordered on 01/15/2024'             -> 'Ordered on <DATE>'


## Complete pipeline function

Wrap all steps into a reusable function.

In [11]:
def clean_text(series, lowercase=True, remove_punct=False, keep_emojis=True):
    # Apply the full text cleaning pipeline
    s = series.map(nfc)
    s = s.map(strip_html)
    s = s.map(mask)
    s = s.map(squash_ws)
    if not keep_emojis:
        s = s.map(lambda x: re.sub(r'[\U0001F600-\U0001F9FF\U00002702-\U000027B0\U0001F300-\U0001F5FF]', '', x))
    if remove_punct:
        s = s.map(remove_punct)
    if lowercase:
        s = s.str.lower()
    return s.reset_index(drop=True)

result = clean_text(raw, lowercase=True, remove_punct=False)
for i, (r, c) in enumerate(zip(raw, result)):
    print(f"{i:2d}: {r!r:55s} -> {c!r}")

 0: '  Loved THE product!! Highly recommend :)  '           -> 'loved the product!! highly recommend :)'
 1: 'I dont like it... too expensive'                       -> 'i dont like it... too expensive'
 2: 'AMAZING!!! Worth every penny.'                         -> 'amazing!!! worth every penny.'
 3: 'Visit https://example.com for more, email me at user@example.com' -> 'visit <url> for more, email me at <email>'
 4: 'cafe was great'                                        -> 'cafe was great'
 5: '<p>HTML <b>markup</b> in the text</p>'                 -> 'html markup in the text'
 6: 'Loved THE product!! Highly recommend :)'               -> 'loved the product!! highly recommend :)'


## Decisions left to you

Text cleaning is **task-dependent**. Here are the key trade-offs:

| Decision | Keep | Remove |
|----------|------|--------|
| **Lowercase** | NER, entity matching | Most classification tasks |
| **Punctuation** | Sentiment analysis ("!!!" is a signal) | TF-IDF, bag-of-words |
| **Stop words** | Large transformer models (BERT handles them) | Small models with limited vocab |
| **Stemming/Lemmatization** | Reduces vocabulary size | Transformers already handle word forms |
| **Emojis** | Social media sentiment | Technical documents |
| **Numbers** | Fraud detection, pricing analysis | General text classification |

**Rule of thumb**: be conservative. It is easier to strip information later than to recover it.